# FrugalProver — budget labeling with a local open model

Runs the oracle pipeline's first two stages end to end:

1. **Stage 1 `sample`** — draw a balanced problem set from MATH.
2. **Stage 2 `budget`** — label each problem with the solve effort it needs, by
   running the **solving agent** (`agent/` — prover → verifier → corrector) once
   per token budget and recording the smallest budget that clears the success
   threshold (`b_star`).

The agent generates locally through `agent/model.py:HFClient` — no server, no
inference-time network. `configs/agent/DeepSeek_R1_Distill_Qwen_7B.yaml` runs the verify-repair loop on
`DeepSeek-R1-Distill-Qwen-7B`; a CUDA GPU is used when present (bf16/fp16),
otherwise it falls back to CPU. Same-model roles share **one** loaded copy.

> **Memory:** the 7B verify-repair loop wants a roomy GPU (A100/L4). On a smaller
> card, edit the `model:` in `configs/agent/DeepSeek_R1_Distill_Qwen_7B.yaml` to a lighter checkpoint
> (e.g. `Qwen/Qwen2.5-1.5B-Instruct`) — since same-model roles share one copy,
> changing it in one place shrinks the whole loop.

## 1. Install (GPU extra pulls torch + transformers)

In [ ]:
# Clone the repo (or run from an existing checkout) and install the GPU extra,
# which pulls torch + transformers (the base install is deliberately torch-free).
!git clone https://github.com/<your-org>/smiles-frugalprover.git
%cd smiles-frugalprover
!pip install -q -e ".[gpu]"

In [ ]:
# Confirm the GPU is visible and the package imports without a torch hoist.
!frugalprover info

## 2. Stage 1 — sample problems from MATH

`frugalprover sample` draws a balanced set and writes `data/<run>/problems.jsonl`
(A1 records). Shrunk here to a handful for a quick demo; drop the `--set`
overrides for the full balanced draw. All later stages resolve files inside the
same run dir, so keep `--run-name` consistent.

In [ ]:
!frugalprover sample -c configs/base.yaml --run-name DeepSeek_R1_Distill_Qwen_7B \
  --set "sample.subjects=[algebra, number_theory]" \
  --set sample.per_level_per_subject=2 \
  --set sample.n_problems=6

## 3. Stage 2 — generate budgets with the agent

`frugalprover budget` (estimator `sweep`, the default) runs the
`DeepSeek_R1_Distill_Qwen_7B.yaml` agent once per budget in `budget.budgets`,
grades the completions against the gold answer, and writes
`data/<run>/budgets.jsonl` (A2).

The budgets come from the agent config (`[8192, 16384, 32768]`) — sized to a
reasoning-model verify-repair loop, where a single solve is already thousands of
tokens. Each role's per-call cap is clamped to the budget, and the loop stops
adding rounds once an attempt's running total reaches it (checked at round
boundaries). Only `n_samples` is trimmed here to keep the demo cheaper.

This is a heavy run (a 7B loop generating tens of thousands of tokens per
problem). It's **resumable** — rerun after a disconnect and it labels only
what's missing.

In [ ]:
# Uses the budgets from the agent config ([8192, 16384, 32768]); only trims
# n_samples to keep the demo cheaper. Add --set "budget.budgets=[...]" to change
# the sweep, but keep entries >= the prover's per-call cap (8192) so B buys
# critique-repair rounds instead of just truncating one attempt.
!frugalprover budget -c configs/base.yaml -c configs/agent/DeepSeek_R1_Distill_Qwen_7B.yaml --run-name DeepSeek_R1_Distill_Qwen_7B \
  --set budget.n_samples=2

## 4. Inspect the budget labels

`b_star` is the smallest budget that cleared the threshold (`null` = censored,
not solved within any budget). `p` is the success rate per budget, `sc` the
self-consistency (plurality-vote) baseline, `tokens_spent` the total the agent
actually generated across the sweep.

In [ ]:
from frugalprover.common.config import load_config
from frugalprover.common.io import read_jsonl

cfg = load_config(["configs/base.yaml", "configs/agent/DeepSeek_R1_Distill_Qwen_7B.yaml"], overrides=["run_name=DeepSeek_R1_Distill_Qwen_7B"])
for row in read_jsonl(cfg.data_path("budgets.jsonl")):
    print(f"[{row['id']}] b_star={row['b_star']} p={row['p']} "
          f"sc={row['sc']} tokens_spent={row['tokens_spent']}")

## 5. (Optional) drive `HFClient` directly

`HFClient` is a thin text-in / text-out interface — handy for a quick sanity
check outside the pipeline. Point `model` at whatever `DeepSeek_R1_Distill_Qwen_7B.yaml` uses.

In [ ]:
from frugalprover.agent.model import build_model_client
from frugalprover.common.config import ModelSpec

client = build_model_client(ModelSpec(client="hf", model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"))
client.setup()
out = client.generate(
    ["Problem: Compute 17 * 23. Give the answer in \\boxed{}."],
    max_tokens=512, temperature=0.6, top_p=0.95, role="prover",
)
print(out[0])
client.teardown()